# M-Shots Learning

In this notebook, we'll explore small prompt engineering techniques and recommendations that will help us elicit responses from the models that are better suited to our needs.

In [1]:
%pip install openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
from getpass import getpass

api_key = getpass("Paste your OpenAI API key here: ")

Path(".env").write_text(f"OPENAI_API_KEY={api_key.strip()}\n")
Path(".gitignore").write_text(".ipynb_checkpoints/\n.env\n__pycache__/\n")

print("Done: .env and .gitignore were created.")

Paste your OpenAI API key here:  ········


Done: .env and .gitignore were created.


In [2]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv

# Load the API key from a local .env file if it exists.
# The .env file should contain a line like:
# OPENAI_API_KEY=your_key_here
#
# IMPORTANT:
# Do not upload your .env file to GitHub.
# The API key is private.
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# I keep the model in one variable so it is easy to change later.
# If the teacher asks for another model, I only need to change this line.
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

In [3]:
print("API key loaded:", OPENAI_API_KEY is not None)

API key loaded: True


# Formatting the answer with Few Shot Samples.

To obtain the model's response in a specific format, we have various options, but one of the most convenient is to use Few-Shot Samples. This involves presenting the model with pairs of user queries and example responses.

Large models like GPT-3.5 respond well to the examples provided, adapting their response to the specified format.

Depending on the number of examples given, this technique can be referred to as:
* Zero-Shot.
* One-Shot.
* Few-Shots.

With One Shot should be enough, and it is recommended to use a maximum of six shots. It's important to remember that this information is passed in each query and occupies space in the input prompt.



In [4]:
# Function to call the model.
def return_OAIResponse(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=1,
        )

    return (response.choices[0].message.content)

In this zero-shots prompt we obtain a correct response, but without formatting, as the model incorporates the information he wants.

In [5]:
#zero-shot
context_user = [
    {'role':'system', 'content':'You are an expert in F1.'}
]
print(return_OAIResponse("Who won the F1 2010?", context_user))

Sebastian Vettel, driving for Red Bull Racing, won the Formula 1 World Championship in 2010.


For a model as large and good as GPT 3.5, a single shot is enough to learn the output format we expect.


In [6]:
#one-shot
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2000 f1 championship?
     Driver: Michael Schumacher.
     Team: Ferrari."""}
]
print(return_OAIResponse("Who won the F1 2011?", context_user))

Driver: Sebastian Vettel.
Team: Red Bull Racing.


Smaller models, or more complicated formats, may require more than one shot. Here a sample with two shots.

In [7]:
#Few shots
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Driver: Fernando Alonso
Team: Renault


In [8]:
print(return_OAIResponse("Who won the F1 2019?", context_user))

The 2019 F1 championship was won by Lewis Hamilton from Mercedes.


We've been creating the prompt without using OpenAI's roles, and as we've seen, it worked correctly.

However, the proper way to do this is by using these roles to construct the prompt, making the model's learning process even more effective.

By not feeding it the entire prompt as if they were system commands, we enable the model to learn from a conversation, which is more realistic for it.

In [9]:
#Recomended solution
context_user = [
    {'role':'system', 'content':'You are and expert in f1.\n\n'},
    {'role':'user', 'content':'Who won the 2010 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Sebastian Vettel. \nTeam: Red Bull. \nPoints: 256. """},
    {'role':'user', 'content':'Who won the 2009 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Jenson Button. \nTeam: BrawnGP. \nPoints: 95. """},
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton. 
Team: Mercedes. 
Points: 413.


We could also address it by using a more conventional prompt, describing what we want and how we want the format.

However, it's essential to understand that in this case, the model is following instructions, whereas in the case of use shots, it is learning in real-time during inference.

In [10]:
context_user = [
    {'role':'system', 'content':"""You are and expert in f1.
    You are going to answer the question of the user giving the name of the rider,
    the name of the team and the points of the champion, following the format:
    Drive:
    Team:
    Points: """
    }
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton
Team: Mercedes
Points: 413


In [11]:
context_user = [
    {'role':'system', 'content':
     """You are classifying .

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Fernando Alonso won the Formula 1 championship in 2006.


Few Shots for classification.


In [12]:
context_user = [
    {'role':'system', 'content':
     """You are an expert in reviewing product opinions and classifying them as positive or negative.

     It fulfilled its function perfectly, I think the price is fair, I would buy it again.
     Sentiment: Positive

     It didn't work bad, but I wouldn't buy it again, maybe it's a bit expensive for what it does.
     Sentiment: Negative.

     I wouldn't know what to say, my son uses it, but he doesn't love it.
     Sentiment: Neutral
     """}
]
print(return_OAIResponse("I'm not going to return it, but I don't plan to buy it again.", context_user))

Sentiment: Neutral


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [13]:
# VERSION 1: ZERO-SHOT PROMPT
# In zero-shot prompting, I give instructions but no examples.
# The model has to understand the task only from the instruction.

customer_message = "My package was supposed to arrive yesterday, but I still have not received anything."

context_zero_shot = [
    {
        "role": "system",
        "content": """
You classify customer messages for an online store.

Choose exactly one category from this list:
- Delivery Issue
- Refund Request
- Product Question
- Positive Feedback
- Complaint

Answer using this format:
Category: <category>
Reason: <short reason>
"""
    }
]

print(return_OAIResponse(customer_message, context_zero_shot))


Category: Delivery Issue
Reason: Customer is inquiring about the status of their package and expressing concern about it not arriving on time.


In [14]:
# VERSION 3: FEW-SHOT PROMPT
# In few-shot prompting, I give several examples.
# This usually gives more stable answers because the model has more patterns to follow.

customer_message = "The headphones are amazing. The sound quality is great and the delivery was fast."

context_few_shot = [
    {
        "role": "system",
        "content": """
You classify customer messages for an online store.

Possible categories:
- Delivery Issue
- Refund Request
- Product Question
- Positive Feedback
- Complaint

Always answer with exactly two lines:
Category: <category>
Reason: <short reason>
"""
    },
    {
        "role": "user",
        "content": "My package arrived broken and the box was open."
    },
    {
        "role": "assistant",
        "content": "Category: Complaint\nReason: The customer received a damaged package."
    },
    {
        "role": "user",
        "content": "Can you tell me if this laptop has 16GB of RAM?"
    },
    {
        "role": "assistant",
        "content": "Category: Product Question\nReason: The customer is asking about a product specification."
    },
    {
        "role": "user",
        "content": "I returned the item last week and I still have not received my money."
    },
    {
        "role": "assistant",
        "content": "Category: Refund Request\nReason: The customer is asking about money after a return."
    },
    {
        "role": "user",
        "content": "The product works perfectly and I would buy it again."
    },
    {
        "role": "assistant",
        "content": "Category: Positive Feedback\nReason: The customer is satisfied with the product."
    }
]

print(return_OAIResponse(customer_message, context_few_shot))


Category: Positive Feedback
Reason: The customer is giving positive feedback on the product and delivery.
